## Dynamic Capabilities

In [0]:
dbutils.widgets.text("file_name", "")

In [0]:
p_file_name = dbutils.widgets.get("file_name")

## Reading data

In [0]:
df = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "parquet")\
                    .option("cloudFiles.schemaLocation", f"abfss://bronze@databrickspipeline07.dfs.core.windows.net/checkpoint_{p_file_name}")\
                    .load(f"abfss://source@databrickspipeline07.dfs.core.windows.net/{p_file_name}")

## Writing data

- processingTime="10 seconds" -> it will run automatically after 10 seconds
- once=True -> it will read all the files that were not processed before, and perform that particular writing part, and stop there. Even if we set this true, only the new data is processed becoz of rocksdb

In [0]:
df.writeStream.format("parquet")\
            .option("checkpointLocation", f"abfss://bronze@databrickspipeline07.dfs.core.windows.net/checkpoint_{p_file_name}")\
            .option("path", f"abfss://bronze@databrickspipeline07.dfs.core.windows.net/{p_file_name}")\
            .trigger(once=True)\
            .start()

In [0]:
df = spark.read.format("parquet")\
            .load(f"abfss://bronze@databrickspipeline07.dfs.core.windows.net/{p_file_name}")
df.display()